# Unified Plotting for U and T Experiments

This notebook processes and plots zonal mean zonal wind (U) and minimum temperature (T) data from multiple experiments. 

Each figure contains two panels: the upper panel shows U and the lower panel shows T. 

The notebook uses unified plotting functions that allow an arbitrary number of experimental groups (from 1 up to 10). 

For each experiment, the following properties are specified:
- **label**: Name of the experiment (including year and perturbation type).
- **x_offset**: The starting x coordinate (e.g. 0 for January, 31 for February, 59 for March, 89 for April).
- **line_color**: Color of the mean line.
- **fill_color**: Color of the envelope (shaded area between minimum and maximum values).

The base reference and climatology curves are plotted in thick black lines (solid for reference and dashed for climatology) and include the base year information. 

Additionally, for the temperature plots, a chlorine activation threshold line (at 197K) is drawn.

The data are read from various directories. For example, the 0008 experiments are located under `/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/` while the NOCOUPL data are also available under `/home/weiji/restart_exam/nocouple_data/`. 

Below, we process the base data and all experiment data, and then produce 20 plots as specified.

In [ ]:
import xarray as xr
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("ticks")

# Suppress warnings if needed
import warnings
warnings.filterwarnings('ignore')

## Data Processing Functions

The following functions read and process the data for U and T. For U, the variable `U` is averaged along longitude and selected at a given latitude. 

For T, the variable `T` is averaged along longitude and the minimum over a specified latitude range is taken.

In [ ]:
def process_U_data(file_path, lat=60, plev=1000):
    """
    Process U data from the given file path.
    
    Parameters:
        file_path (str): Path with wildcard to the netCDF files.
        lat (int/float): Latitude to select.
        plev (int): Pressure level (in Pa) to select.
    
    Returns:
        xarray.DataArray: Processed U data.
    """
    ds = xr.open_mfdataset(file_path, concat_dim='member', combine='nested')
    U = ds['U']
    U = U.mean(dim='lon')
    U = U.sel(plev=plev)
    var = U.sel(lat=lat, method='nearest')
    return var


def process_T_data(file_path, plev=5000, lat_range=(60, 90)):
    """
    Process T data from the given file path.
    
    Parameters:
        file_path (str): Path with wildcard to the netCDF files.
        plev (int): Pressure level (in Pa) to select.
        lat_range (tuple): Latitude range (min_lat, max_lat) to consider.
    
    Returns:
        xarray.DataArray: Processed T data (minimum over the selected latitude range).
    """
    ds = xr.open_mfdataset(file_path, concat_dim='member', combine='nested')
    T = ds['T']
    T = T.mean(dim='lon')
    T = T.sel(plev=plev)
    var = T.sel(lat=slice(lat_range[0], lat_range[1])).min(dim='lat')
    return var


def process_U_base(file_path, lat=60, plev=1000, months=[1,2,3,4,5,6], base_year='0008'):
    """
    Process the base U data.
    
    Parameters:
        file_path (str): Path to the netCDF file.
        lat (int/float): Specified latitude.
        plev (int): Pressure level (Pa).
        months (list): List of months to include.
        base_year (str): Year to select for the reference curve.
        
    Returns:
        ref (xarray.DataArray): Reference U data for the specified base_year.
        clim (xarray.DataArray): Climatology of U data.
    """
    ds = xr.open_dataset(file_path)
    U = ds['U']
    U = U.sel(plev=plev)
    U = U.sel(time=ds.time.dt.month.isin(months))
    U = U.mean(dim='lon')
    var = U.sel(lat=lat, method='nearest')
    
    # Use the specified base_year instead of hard-coded 8
    ref = var.sel(time=var.time.dt.year == int(base_year))
    clim = var.groupby("time.dayofyear").mean()
    return ref, clim



# Modified process_T_base function with a base_year parameter
def process_T_base(file_path, plev=5000, lat_range=(60,90), months=[1,2,3,4,5,6], base_year='0008'):
    """
    Process the base T data.
    
    Parameters:
        file_path (str): Path to the netCDF file.
        plev (int): Pressure level (Pa).
        lat_range (tuple): Latitude range (min_lat, max_lat).
        months (list): List of months to include.
        base_year (str): Year to select for the reference curve.
        
    Returns:
        ref (xarray.DataArray): Reference T data for the specified base_year.
        clim (xarray.DataArray): Climatology of T data.
    """
    ds = xr.open_dataset(file_path)
    T = ds['T']
    T = T.sel(plev=plev)
    T = T.sel(time=ds.time.dt.month.isin(months))
    T = T.mean(dim='lon')
    var = T.sel(lat=slice(lat_range[0], lat_range[1])).min(dim='lat')
    
    # Use the specified base_year instead of hard-coded 8
    ref = var.sel(time=var.time.dt.year == int(base_year))
    clim = var.groupby("time.dayofyear").mean()
    return ref, clim


## Unified Plotting Function for U and T

This function creates a figure with two panels: the upper panel plots U and the lower panel plots T. 

Parameters:
- **base_ref_U** and **base_clim_U**: Base reference and climatology data for U.
- **base_ref_T** and **base_clim_T**: Base reference and climatology data for T.
- **experiments**: A list of dictionaries. Each dictionary must contain:
    - `label`: Experiment label (string).
    - `U`: Processed U data (xarray.DataArray).
    - `T`: Processed T data (xarray.DataArray).
    - `x_offset`: Starting x coordinate (integer).
    - `line_color`: Color for the mean line.
    - `fill_color`: Color for the envelope (shaded area).
- **base_year**: Year string for the base data (e.g., "0008").
- **save_path**: Path (without extension) to save the figure.
- **lat**, **plev_U**: Parameters for U label.
- **lat_range**, **plev_T**: Parameters for T label.

A chlorine activation threshold line is added in the T panel.

In [ ]:
def plot_experiments_UT(base_ref_U, base_clim_U, base_ref_T, base_clim_T, experiments, base_year,
                        save_path, lat=60, plev_U=1000, lat_range=(60,90), plev_T=5000):
    """
    Create a figure with two panels (U and T) and plot base curves along with experiment data.
    
    Parameters:
        base_ref_U, base_clim_U: Base U reference and climatology data.
        base_ref_T, base_clim_T: Base T reference and climatology data.
        experiments (list): List of dictionaries containing experiment data and plotting parameters.
        base_year (str): Year for the base curves (e.g., "0008").
        save_path (str): File path (without extension) to save the figure.
        lat (int/float): Latitude used for U plots.
        plev_U (int): Pressure level (Pa) for U.
        lat_range (tuple): Latitude range for T plots.
        plev_T (int): Pressure level (Pa) for T.
    
    Returns:
        fig, ax: Matplotlib figure and axes array.
    """
    # Create a figure with 2 subplots (U on top, T on bottom)
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 16), constrained_layout=True)
    
    # Plot base U curves
    x_base_U = range(len(base_ref_U))
    ax1.plot(x_base_U, base_ref_U, color='black', linewidth=5, label=f'Base {base_year}')
    x_clim_U = range(len(base_clim_U))
    ax1.plot(x_clim_U, base_clim_U, color='black', linestyle='--', linewidth=5, label=f'Clim {base_year}')
    
    # Plot experiment U data
    for exp in experiments:
        # x coordinate based on x_offset and length of time dimension
        x_vals = range(exp['x_offset'], exp['x_offset'] + len(exp['U'].time))
        mean_U = exp['U'].mean(dim='member')
        env_min_U = exp['U'].min(dim='member')
        env_max_U = exp['U'].max(dim='member')
        ax1.plot(x_vals, mean_U, color=exp['line_color'], linewidth=3, label=exp['label'])
        ax1.fill_between(x_vals, env_min_U, env_max_U, color=exp['fill_color'], alpha=0.3)
    
    ax1.set_xlim(0, 150)
    ax1.set_xticks([0, 31, 59, 89, 120])
    ax1.set_xticklabels(['Jan','Feb','Mar','Apr','May'])
    ax1.set_ylabel(f'Zonal Mean Wind, {lat}°, {plev_U/100:.0f} hPa (m/s)', fontsize=14)
    ax1.legend(fontsize=12)
    ax1.set_ylim(-40, 80)
    
    # Plot base T curves
    x_base_T = range(len(base_ref_T))
    ax2.plot(x_base_T, base_ref_T, color='black', linewidth=5, label=f'Base {base_year}')
    x_clim_T = range(len(base_clim_T))
    ax2.plot(x_clim_T, base_clim_T, color='black', linestyle='--', linewidth=5, label=f'Clim {base_year}')
    
    # Plot experiment T data
    for exp in experiments:
        x_vals = range(exp['x_offset'], exp['x_offset'] + len(exp['T'].time))
        mean_T = exp['T'].mean(dim='member')
        env_min_T = exp['T'].min(dim='member')
        env_max_T = exp['T'].max(dim='member')
        ax2.plot(x_vals, mean_T, color=exp['line_color'], linewidth=3, label=exp['label'])
        ax2.fill_between(x_vals, env_min_T, env_max_T, color=exp['fill_color'], alpha=0.3)
    
    # Add chlorine activation threshold line in T panel
    ax2.axhline(y=197, color='royalblue', linestyle='--', linewidth=2)
    ax2.text(5, 194, 'Cl activation threshold', horizontalalignment='left', fontsize=12, color='royalblue')
    
    ax2.set_xlim(0, 150)
    ax2.set_xticks([0, 31, 59, 89, 120])
    ax2.set_xticklabels(['Jan','Feb','Mar','Apr','May'])
    ax2.set_ylabel(f'Minimum Temperature, {lat_range[0]}-{lat_range[1]}°N, {plev_T/100:.0f} hPa (K)', fontsize=14)
    ax2.set_xlabel('Day of Year', fontsize=14)
    ax2.legend(fontsize=12)
    ax2.set_ylim(170, 235)
    
    # Save the figure as PDF and PNG
    plt.savefig(f'{save_path}.pdf', bbox_inches='tight')
    plt.savefig(f'{save_path}.png', bbox_inches='tight')
    
    return fig, (ax1, ax2)

## Define Base Data Paths and Process Base Data

The base datasets (for both U and T) come from the merged file.

## Process Experiment Data

Below, we process all experiment data for both U and T. The naming convention is as follows:

- **0008Jan_small_pert**: January small perturbation for 0008 (x_offset = 0).
- **0008Feb_small_pert**: February small perturbation for 0008 (x_offset = 31).
- **0008Mar_small_pert**: March small perturbation for 0008 (x_offset = 59).

Similarly for other experiments. For NOCOUPL data, the ones in `/mnt/.../FNC` are read as reference (e.g. `0008Feb_NOCOUPL_reference`), and the ones in `/home/weiji/restart_exam/nocouple_data/` are used for plotting.

All file paths use a wildcard pattern (`*.nc`).

In [ ]:
# --- 0008 Experiments ---
# Base file path for year 0008
base_file = '/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.merged.nc'


# 0008 Jan small perturbation (x_offset = 0)
data_0008Jan_U = process_U_data('/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.002_0008/Jan/*.nc', lat=60, plev=1000)
data_0008Jan_T = process_T_data('/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.002_0008/Jan/*.nc', plev=5000, lat_range=(60,90))

# 0008 Feb small perturbation (x_offset = 31)
data_0008Feb_small_U = process_U_data('/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.002_0008/Feb/*.nc', lat=60, plev=1000)
data_0008Feb_small_T = process_T_data('/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.002_0008/Feb/*.nc', plev=5000, lat_range=(60,90))

# 0008 Mar small perturbation (x_offset = 59)
data_0008Mar_small_U = process_U_data('/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.002_0008/Mar/*.nc', lat=60, plev=1000)
data_0008Mar_small_T = process_T_data('/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.002_0008/Mar/*.nc', plev=5000, lat_range=(60,90))

# 0008 Feb large perturbation (x_offset = 31)
data_0008Feb_large_U = process_U_data('/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.002_0008/Feb_2/*.nc', lat=60, plev=1000)
data_0008Feb_large_T = process_T_data('/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.002_0008/Feb_2/*.nc', plev=5000, lat_range=(60,90))

# 0008 Mar large perturbation (x_offset = 59)
data_0008Mar_large_U = process_U_data('/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.002_0008/Mar_2/*.nc', lat=60, plev=1000)
data_0008Mar_large_T = process_T_data('/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.002_0008/Mar_2/*.nc', plev=5000, lat_range=(60,90))

# 0008 Apr large perturbation (x_offset = 89)
data_0008Apr_large_U = process_U_data('/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.002_0008/Apr/*.nc', lat=60, plev=1000)
data_0008Apr_large_T = process_T_data('/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.002_0008/Apr/*.nc', plev=5000, lat_range=(60,90))

# 0008 Feb water vapor perturbation (x_offset = 31)
data_0008Feb_vapor_U = process_U_data('/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.002_0008/Feb_3/*.nc', lat=60, plev=1000)
data_0008Feb_vapor_T = process_T_data('/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.002_0008/Feb_3/*.nc', plev=5000, lat_range=(60,90))

# 0008 Mar water vapor perturbation (x_offset = 59)
data_0008Mar_vapor_U = process_U_data('/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.002_0008/Mar_3/*.nc', lat=60, plev=1000)
data_0008Mar_vapor_T = process_T_data('/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.002_0008/Mar_3/*.nc', plev=5000, lat_range=(60,90))

# 0008 Feb NOCOUPL reference (from FNC folder, x_offset = 31)
data_0008Feb_NOCOUPL_ref_U = process_U_data('/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.002_0008/Feb_NOCOUPL/*.nc', lat=60, plev=1000)
data_0008Feb_NOCOUPL_ref_T = process_T_data('/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.002_0008/Feb_NOCOUPL/*.nc', plev=5000, lat_range=(60,90))

# 0008 Feb NOCOUPL (from nocouple_data folder, x_offset = 31)
data_0008Feb_NOCOUPL_U = process_U_data('/home/weiji/restart_exam/nocouple_data/0008/Feb_NOCOUPL/0008-02/*.nc', lat=60, plev=1000)
data_0008Feb_NOCOUPL_T = process_T_data('/home/weiji/restart_exam/nocouple_data/0008/Feb_NOCOUPL/0008-02/*.nc', plev=5000, lat_range=(60,90))

# 0008 Mar NOCOUPL (x_offset = 59)
data_0008Mar_NOCOUPL_U = process_U_data('/home/weiji/restart_exam/nocouple_data/0008/Mar_NOCOUPL/0008-03/*.nc', lat=60, plev=1000)
data_0008Mar_NOCOUPL_T = process_T_data('/home/weiji/restart_exam/nocouple_data/0008/Mar_NOCOUPL/0008-03/*.nc', plev=5000, lat_range=(60,90))

# --- Other Years (0003, 0013, 0014, 0019) for small perturbation ---

def process_year_small(year, month):
    # month should be either 'Feb' or 'Mar'; x_offset = 31 for Feb and 59 for Mar
    offset = 31 if month == 'Feb' else 59
    base_path = f'/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.002_{year}/{month}/*.nc'
    U_data = process_U_data(base_path, lat=60, plev=1000)
    T_data = process_T_data(base_path, plev=5000, lat_range=(60,90))
    return U_data, T_data, offset

# 0003
data_0003Feb_small_U, data_0003Feb_small_T, off_0003Feb = process_year_small('0003', 'Feb')
data_0003Mar_small_U, data_0003Mar_small_T, off_0003Mar = process_year_small('0003', 'Mar')

# 0013
data_0013Feb_small_U, data_0013Feb_small_T, off_0013Feb = process_year_small('0013', 'Feb')
data_0013Mar_small_U, data_0013Mar_small_T, off_0013Mar = process_year_small('0013', 'Mar')

# 0014
data_0014Feb_small_U, data_0014Feb_small_T, off_0014Feb = process_year_small('0014', 'Feb')
data_0014Mar_small_U, data_0014Mar_small_T, off_0014Mar = process_year_small('0014', 'Mar')

# 0019
data_0019Feb_small_U, data_0019Feb_small_T, off_0019Feb = process_year_small('0019', 'Feb')
data_0019Mar_small_U, data_0019Mar_small_T, off_0019Mar = process_year_small('0019', 'Mar')

# --- NOCOUPL for other years (0013, 0014, 0019) ---

def process_year_nocouple(year, month):
    offset = 31 if month == 'Feb' else 59
    month_code = '02' if month == 'Feb' else '03'
    base_path = f"/home/weiji/restart_exam/nocouple_data/{year}/{month}_NOCOUPL/{year}-{month_code}/*.nc"

    U_data = process_U_data(base_path, lat=60, plev=1000)
    T_data = process_T_data(base_path, plev=5000, lat_range=(60,90))
    return U_data, T_data, offset

# 0013 NOCOUPL
data_0013Feb_NOCOUPL_U, data_0013Feb_NOCOUPL_T, off_0013Feb_noc = process_year_nocouple('0013', 'Feb')
data_0013Mar_NOCOUPL_U, data_0013Mar_NOCOUPL_T, off_0013Mar_noc = process_year_nocouple('0013', 'Mar')

# 0014 NOCOUPL
data_0014Feb_NOCOUPL_U, data_0014Feb_NOCOUPL_T, off_0014Feb_noc = process_year_nocouple('0014', 'Feb')
data_0014Mar_NOCOUPL_U, data_0014Mar_NOCOUPL_T, off_0014Mar_noc = process_year_nocouple('0014', 'Mar')

# 0019 NOCOUPL
data_0019Feb_NOCOUPL_U, data_0019Feb_NOCOUPL_T, off_0019Feb_noc = process_year_nocouple('0019', 'Feb')
data_0019Mar_NOCOUPL_U, data_0019Mar_NOCOUPL_T, off_0019Mar_noc = process_year_nocouple('0019', 'Mar')

## Plotting the Figures

Below, we generate 20 plots as specified. In each plot, the experiments are provided as a list of dictionaries with keys:
  - **label**
  - **U** and **T** data
  - **x_offset** (31 for Feb data, 59 for Mar data, etc.)
  - **line_color** and **fill_color**. 

If the number of experiments in a plot is 2 or 3, default colors (in order: `forestgreen`, `royalblue`, `hotpink`) are used if not specified.

The base curves are always those from the 0008 merged file (base year = "0008").

Each plot is saved as both PDF and PNG.

In [ ]:
# Helper function remains unchanged
def assign_default_colors(exp_list):
    defaults = [('forestgreen', 'forestgreen'), ('royalblue', 'royalblue'), ('hotpink', 'hotpink')]
    for i, exp in enumerate(exp_list):
        if 'line_color' not in exp or not exp['line_color']:
            exp['line_color'] = defaults[i % len(defaults)][0]
        if 'fill_color' not in exp or not exp['fill_color']:
            exp['fill_color'] = defaults[i % len(defaults)][1]
    return exp_list

# Pre-compute base data for each base year to avoid redundant processing
base_years = ["0008", "0003", "0013", "0014", "0019"]
base_data = {}
for year in base_years:
    base_data[year] = {}
    base_data[year]['ref_U'], base_data[year]['clim_U'] = process_U_base(
        base_file, lat=60, plev=1000, months=[1, 2, 3, 4, 5, 6], base_year=year
    )
    base_data[year]['ref_T'], base_data[year]['clim_T'] = process_T_base(
        base_file, plev=5000, lat_range=(60, 90), months=[1, 2, 3, 4, 5, 6], base_year=year
    )

# Plot 1: Validation plot for 0008Feb NOCOUPL_reference and 0008Feb NOCOUPL (base year: 0008)
base_year = "0008"
exps = [
    {'label': '0008Feb_NOCOUPL_reference', 'U': data_0008Feb_NOCOUPL_ref_U, 'T': data_0008Feb_NOCOUPL_ref_T, 'x_offset': 31},
    {'label': '0008Feb_NOCOUPL', 'U': data_0008Feb_NOCOUPL_U, 'T': data_0008Feb_NOCOUPL_T, 'x_offset': 31}
]
exps = assign_default_colors(exps)
plot_experiments_UT(base_data[base_year]['ref_U'], base_data[base_year]['clim_U'],
                    base_data[base_year]['ref_T'], base_data[base_year]['clim_T'],
                    exps, base_year,
                    save_path='/home/weiji/restart_exam/plots/Validation_0008Feb_NOCOUPL')

# Plot 2: 0008Jan small, 0008Feb small, 0008Mar small (base year: 0008)
base_year = "0008"
exps = [
    {'label': '0008Jan_small_pert', 'U': data_0008Jan_U, 'T': data_0008Jan_T, 'x_offset': 0},
    {'label': '0008Feb_small_pert', 'U': data_0008Feb_small_U, 'T': data_0008Feb_small_T, 'x_offset': 31},
    {'label': '0008Mar_small_pert', 'U': data_0008Mar_small_U, 'T': data_0008Mar_small_T, 'x_offset': 59}
]
exps = assign_default_colors(exps)
plot_experiments_UT(base_data[base_year]['ref_U'], base_data[base_year]['clim_U'],
                    base_data[base_year]['ref_T'], base_data[base_year]['clim_T'],
                    exps, base_year,
                    save_path='/home/weiji/restart_exam/plots/0008_small_pert')

# Plot 3: 0008Feb large, 0008Mar large, 0008Apr large (base year: 0008)
base_year = "0008"
exps = [
    {'label': '0008Feb_large_pert', 'U': data_0008Feb_large_U, 'T': data_0008Feb_large_T, 'x_offset': 31},
    {'label': '0008Mar_large_pert', 'U': data_0008Mar_large_U, 'T': data_0008Mar_large_T, 'x_offset': 59},
    {'label': '0008Apr_large_pert', 'U': data_0008Apr_large_U, 'T': data_0008Apr_large_T, 'x_offset': 89}
]
exps = assign_default_colors(exps)
plot_experiments_UT(base_data[base_year]['ref_U'], base_data[base_year]['clim_U'],
                    base_data[base_year]['ref_T'], base_data[base_year]['clim_T'],
                    exps, base_year,
                    save_path='/home/weiji/restart_exam/plots/0008_large_pert')

# Plot 4: 0008Feb small, 0008Feb large, 0008Feb water vapor (all February, base year: 0008)
base_year = "0008"
exps = [
    {'label': '0008Feb_small_pert', 'U': data_0008Feb_small_U, 'T': data_0008Feb_small_T, 'x_offset': 31},
    {'label': '0008Feb_large_pert', 'U': data_0008Feb_large_U, 'T': data_0008Feb_large_T, 'x_offset': 31},
    {'label': '0008Feb_vapor_pert', 'U': data_0008Feb_vapor_U, 'T': data_0008Feb_vapor_T, 'x_offset': 31}
]
exps = assign_default_colors(exps)
plot_experiments_UT(base_data[base_year]['ref_U'], base_data[base_year]['clim_U'],
                    base_data[base_year]['ref_T'], base_data[base_year]['clim_T'],
                    exps, base_year,
                    save_path='/home/weiji/restart_exam/plots/0008_Feb_pert_combo')

# Plot 5: 0003Feb small, 0003Mar small (base year: 0003)
base_year = "0003"
exps = [
    {'label': '0003Feb_small_pert', 'U': data_0003Feb_small_U, 'T': data_0003Feb_small_T, 'x_offset': off_0003Feb},
    {'label': '0003Mar_small_pert', 'U': data_0003Mar_small_U, 'T': data_0003Mar_small_T, 'x_offset': off_0003Mar}
]
exps = assign_default_colors(exps)
plot_experiments_UT(base_data[base_year]['ref_U'], base_data[base_year]['clim_U'],
                    base_data[base_year]['ref_T'], base_data[base_year]['clim_T'],
                    exps, base_year,
                    save_path='/home/weiji/restart_exam/plots/0003_small_pert')

# Plot 6: 0013Feb small, 0013Mar small (base year: 0013)
base_year = "0013"
exps = [
    {'label': '0013Feb_small_pert', 'U': data_0013Feb_small_U, 'T': data_0013Feb_small_T, 'x_offset': off_0013Feb},
    {'label': '0013Mar_small_pert', 'U': data_0013Mar_small_U, 'T': data_0013Mar_small_T, 'x_offset': off_0013Mar}
]
exps = assign_default_colors(exps)
plot_experiments_UT(base_data[base_year]['ref_U'], base_data[base_year]['clim_U'],
                    base_data[base_year]['ref_T'], base_data[base_year]['clim_T'],
                    exps, base_year,
                    save_path='/home/weiji/restart_exam/plots/0013_small_pert')

# Plot 7: 0014Feb small, 0014Mar small (base year: 0014)
base_year = "0014"
exps = [
    {'label': '0014Feb_small_pert', 'U': data_0014Feb_small_U, 'T': data_0014Feb_small_T, 'x_offset': off_0014Feb},
    {'label': '0014Mar_small_pert', 'U': data_0014Mar_small_U, 'T': data_0014Mar_small_T, 'x_offset': off_0014Mar}
]
exps = assign_default_colors(exps)
plot_experiments_UT(base_data[base_year]['ref_U'], base_data[base_year]['clim_U'],
                    base_data[base_year]['ref_T'], base_data[base_year]['clim_T'],
                    exps, base_year,
                    save_path='/home/weiji/restart_exam/plots/0014_small_pert')

# Plot 8: 0019Feb small, 0019Mar small (base year: 0019)
base_year = "0019"
exps = [
    {'label': '0019Feb_small_pert', 'U': data_0019Feb_small_U, 'T': data_0019Feb_small_T, 'x_offset': off_0019Feb},
    {'label': '0019Mar_small_pert', 'U': data_0019Mar_small_U, 'T': data_0019Mar_small_T, 'x_offset': off_0019Mar}
]
exps = assign_default_colors(exps)
plot_experiments_UT(base_data[base_year]['ref_U'], base_data[base_year]['clim_U'],
                    base_data[base_year]['ref_T'], base_data[base_year]['clim_T'],
                    exps, base_year,
                    save_path='/home/weiji/restart_exam/plots/0019_small_pert')

# Plot 9: 0008 NOCOUPL experiments (base year: 0008)
base_year = "0008"
exps = [
    {'label': '0008Feb_NOCOUPL', 'U': data_0008Feb_NOCOUPL_U, 'T': data_0008Feb_NOCOUPL_T, 'x_offset': 31},
    {'label': '0008Mar_NOCOUPL', 'U': data_0008Mar_NOCOUPL_U, 'T': data_0008Mar_NOCOUPL_T, 'x_offset': 59}
]
exps = assign_default_colors(exps)
plot_experiments_UT(base_data[base_year]['ref_U'], base_data[base_year]['clim_U'],
                    base_data[base_year]['ref_T'], base_data[base_year]['clim_T'],
                    exps, base_year,
                    save_path='/home/weiji/restart_exam/plots/0008_nocouple')

# Plot 10: 0013 NOCOUPL experiments (base year: 0013)
base_year = "0013"
exps = [
    {'label': '0013Feb_NOCOUPL', 'U': data_0013Feb_NOCOUPL_U, 'T': data_0013Feb_NOCOUPL_T, 'x_offset': off_0013Feb_noc},
    {'label': '0013Mar_NOCOUPL', 'U': data_0013Mar_NOCOUPL_U, 'T': data_0013Mar_NOCOUPL_T, 'x_offset': off_0013Mar_noc}
]
exps = assign_default_colors(exps)
plot_experiments_UT(base_data[base_year]['ref_U'], base_data[base_year]['clim_U'],
                    base_data[base_year]['ref_T'], base_data[base_year]['clim_T'],
                    exps, base_year,
                    save_path='/home/weiji/restart_exam/plots/0013_nocouple')

# Plot 11: 0014 NOCOUPL experiments (base year: 0014)
base_year = "0014"
exps = [
    {'label': '0014Feb_NOCOUPL', 'U': data_0014Feb_NOCOUPL_U, 'T': data_0014Feb_NOCOUPL_T, 'x_offset': off_0014Feb_noc},
    {'label': '0014Mar_NOCOUPL', 'U': data_0014Mar_NOCOUPL_U, 'T': data_0014Mar_NOCOUPL_T, 'x_offset': off_0014Mar_noc}
]
exps = assign_default_colors(exps)
plot_experiments_UT(base_data[base_year]['ref_U'], base_data[base_year]['clim_U'],
                    base_data[base_year]['ref_T'], base_data[base_year]['clim_T'],
                    exps, base_year,
                    save_path='/home/weiji/restart_exam/plots/0014_nocouple')

# Plot 12: 0019 NOCOUPL experiments (base year: 0019)
base_year = "0019"
exps = [
    {'label': '0019Feb_NOCOUPL', 'U': data_0019Feb_NOCOUPL_U, 'T': data_0019Feb_NOCOUPL_T, 'x_offset': off_0019Feb_noc},
    {'label': '0019Mar_NOCOUPL', 'U': data_0019Mar_NOCOUPL_U, 'T': data_0019Mar_NOCOUPL_T, 'x_offset': off_0019Mar_noc}
]
exps = assign_default_colors(exps)
plot_experiments_UT(base_data[base_year]['ref_U'], base_data[base_year]['clim_U'],
                    base_data[base_year]['ref_T'], base_data[base_year]['clim_T'],
                    exps, base_year,
                    save_path='/home/weiji/restart_exam/plots/0019_nocouple')

# Plot 13: 0008Feb small vs 0008Feb NOCOUPL (base year: 0008)
base_year = "0008"
exps = [
    {'label': '0008Feb_small_pert', 'U': data_0008Feb_small_U, 'T': data_0008Feb_small_T, 'x_offset': 31},
    {'label': '0008Feb_NOCOUPL', 'U': data_0008Feb_NOCOUPL_U, 'T': data_0008Feb_NOCOUPL_T, 'x_offset': 31}
]
exps = assign_default_colors(exps)
plot_experiments_UT(base_data[base_year]['ref_U'], base_data[base_year]['clim_U'],
                    base_data[base_year]['ref_T'], base_data[base_year]['clim_T'],
                    exps, base_year,
                    save_path='/home/weiji/restart_exam/plots/0008Feb_small_vs_nocouple')

# Plot 14: 0008Mar small vs 0008Mar NOCOUPL (base year: 0008)
base_year = "0008"
exps = [
    {'label': '0008Mar_small_pert', 'U': data_0008Mar_small_U, 'T': data_0008Mar_small_T, 'x_offset': 59},
    {'label': '0008Mar_NOCOUPL', 'U': data_0008Mar_NOCOUPL_U, 'T': data_0008Mar_NOCOUPL_T, 'x_offset': 59}
]
exps = assign_default_colors(exps)
plot_experiments_UT(base_data[base_year]['ref_U'], base_data[base_year]['clim_U'],
                    base_data[base_year]['ref_T'], base_data[base_year]['clim_T'],
                    exps, base_year,
                    save_path='/home/weiji/restart_exam/plots/0008Mar_small_vs_nocouple')

# Plot 15: 0013Feb small vs 0013Feb NOCOUPL (base year: 0013)
base_year = "0013"
exps = [
    {'label': '0013Feb_small_pert', 'U': data_0013Feb_small_U, 'T': data_0013Feb_small_T, 'x_offset': off_0013Feb},
    {'label': '0013Feb_NOCOUPL', 'U': data_0013Feb_NOCOUPL_U, 'T': data_0013Feb_NOCOUPL_T, 'x_offset': off_0013Feb}
]
exps = assign_default_colors(exps)
plot_experiments_UT(base_data[base_year]['ref_U'], base_data[base_year]['clim_U'],
                    base_data[base_year]['ref_T'], base_data[base_year]['clim_T'],
                    exps, base_year,
                    save_path='/home/weiji/restart_exam/plots/0013Feb_small_vs_nocouple')

# Plot 16: 0013Mar small vs 0013Mar NOCOUPL (base year: 0013)
base_year = "0013"
exps = [
    {'label': '0013Mar_small_pert', 'U': data_0013Mar_small_U, 'T': data_0013Mar_small_T, 'x_offset': off_0013Mar},
    {'label': '0013Mar_NOCOUPL', 'U': data_0013Mar_NOCOUPL_U, 'T': data_0013Mar_NOCOUPL_T, 'x_offset': off_0013Mar}
]
exps = assign_default_colors(exps)
plot_experiments_UT(base_data[base_year]['ref_U'], base_data[base_year]['clim_U'],
                    base_data[base_year]['ref_T'], base_data[base_year]['clim_T'],
                    exps, base_year,
                    save_path='/home/weiji/restart_exam/plots/0013Mar_small_vs_nocouple')

# Plot 17: 0014Feb small vs 0014Feb NOCOUPL (base year: 0014)
base_year = "0014"
exps = [
    {'label': '0014Feb_small_pert', 'U': data_0014Feb_small_U, 'T': data_0014Feb_small_T, 'x_offset': off_0014Feb},
    {'label': '0014Feb_NOCOUPL', 'U': data_0014Feb_NOCOUPL_U, 'T': data_0014Feb_NOCOUPL_T, 'x_offset': off_0014Feb}
]
exps = assign_default_colors(exps)
plot_experiments_UT(base_data[base_year]['ref_U'], base_data[base_year]['clim_U'],
                    base_data[base_year]['ref_T'], base_data[base_year]['clim_T'],
                    exps, base_year,
                    save_path='/home/weiji/restart_exam/plots/0014Feb_small_vs_nocouple')

# Plot 18: 0014Mar small vs 0014Mar NOCOUPL (base year: 0014)
base_year = "0014"
exps = [
    {'label': '0014Mar_small_pert', 'U': data_0014Mar_small_U, 'T': data_0014Mar_small_T, 'x_offset': off_0014Mar},
    {'label': '0014Mar_NOCOUPL', 'U': data_0014Mar_NOCOUPL_U, 'T': data_0014Mar_NOCOUPL_T, 'x_offset': off_0014Mar}
]
exps = assign_default_colors(exps)
plot_experiments_UT(base_data[base_year]['ref_U'], base_data[base_year]['clim_U'],
                    base_data[base_year]['ref_T'], base_data[base_year]['clim_T'],
                    exps, base_year,
                    save_path='/home/weiji/restart_exam/plots/0014Mar_small_vs_nocouple')

# Plot 19: 0019Feb small vs 0019Feb NOCOUPL (base year: 0019)
base_year = "0019"
exps = [
    {'label': '0019Feb_small_pert', 'U': data_0019Feb_small_U, 'T': data_0019Feb_small_T, 'x_offset': off_0019Feb},
    {'label': '0019Feb_NOCOUPL', 'U': data_0019Feb_NOCOUPL_U, 'T': data_0019Feb_NOCOUPL_T, 'x_offset': off_0019Feb}
]
exps = assign_default_colors(exps)
plot_experiments_UT(base_data[base_year]['ref_U'], base_data[base_year]['clim_U'],
                    base_data[base_year]['ref_T'], base_data[base_year]['clim_T'],
                    exps, base_year,
                    save_path='/home/weiji/restart_exam/plots/0019Feb_small_vs_nocouple')

# Plot 20: 0019Mar small vs 0019Mar NOCOUPL (base year: 0019)
base_year = "0019"
exps = [
    {'label': '0019Mar_small_pert', 'U': data_0019Mar_small_U, 'T': data_0019Mar_small_T, 'x_offset': off_0019Mar},
    {'label': '0019Mar_NOCOUPL', 'U': data_0019Mar_NOCOUPL_U, 'T': data_0019Mar_NOCOUPL_T, 'x_offset': off_0019Mar}
]
exps = assign_default_colors(exps)
plot_experiments_UT(base_data[base_year]['ref_U'], base_data[base_year]['clim_U'],
                    base_data[base_year]['ref_T'], base_data[base_year]['clim_T'],
                    exps, base_year,
                    save_path='/home/weiji/restart_exam/plots/0019Mar_small_vs_nocouple')
